Simulated Annealing Experiments

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from datetime import datetime

# Parameters (set accordingly)
results_path = f'data/simulated_annealing/summary_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.npy'
brute_force_data_directory = './data/brute_force/first_run'

num_runs = 1000
init_temp = 1000
end_temp = 1
max_temp_iterations = 512

In [9]:
import numpy as np

temp_iterations = np.geomspace(10, max_temp_iterations, num=40, endpoint=True, dtype=int)

In [10]:
np.random.seed(4321) # set global seed for reproducibility

In [11]:
from pathlib import Path
import pandas as pd
import re

# Extract information from brute force data
extract_candidate_dimension = lambda p: np.log2(int(re.search(string=str(p.name), pattern=r'_(\d*)_candidates').group(1)))
extract_load_factor = lambda p: float(re.search(string=str(p.name), pattern=r'_(\d*\.\d*)_load_factor').group(1))

data_files = [p for p in Path(brute_force_data_directory).glob("*.csv") if p.is_file()]
candidate_dims = [(extract_candidate_dimension(p), extract_load_factor(p), p) for p in data_files]

df = pd.DataFrame(data=candidate_dims, columns=['dim', 'load factor', 'path'])
df = df.groupby(['dim', 'load factor'])['path'].agg(list).reset_index()

In [12]:
problem_sizes = df['dim'].unique()
load_factors = df['load factor'].unique()
brute_force_data_paths = np.array(df['path'].to_list()).reshape(len(problem_sizes), len(load_factors), -1)

In [ ]:
import pandas as pd
import numpy as np
brute_force_data = np.vectorize(pd.read_csv)(brute_force_data_paths)
loss_spaces = [np.array([d['loss'] for d in da]) for data in brute_force_data for da in data] # needs to be list because is inhomogeneous

In [65]:
from cl_optimizer import SimulatedAnnealing

def create_simulated_annealing_instances(x: np.ndarray):
    return np.array([
        SimulatedAnnealing(
            lookup_table=data_path,
            bounds=int(x[0]), # problem size
        )  for data_path in x[1:]])

sim_annss = np.array(
    [
        np.apply_along_axis(func1d=create_simulated_annealing_instances, arr=np.column_stack((problem_sizes, brute_force_2d)), axis=1)
        for brute_force_2d in np.swapaxes(brute_force_data, 0, 1) # need to swap axes here
    ]
) # looks weird but just creates a simulated annealing object per problem size, load factor and instance
sim_annss = np.swapaxes(sim_annss, 0, 1) # swap axes back

In [68]:
from parallelization import get_execute_sim_ann

execute_simann_fun = get_execute_sim_ann(init_temp, end_temp)

def execute_in_parallel(problem_size: int, sas_per_load_factor: list):
    resultsss = []
    for j, sas_per_instance in enumerate(sas_per_load_factor):
        resultss = []
        for k, sim_ann in enumerate(sas_per_instance):
            results = []
            for l in range(num_runs):
                print(f"Problem size: {problem_size + 1}/{int(problem_sizes[-1])} - load factor: {j + 1}/{len(sas_per_load_factor)} - instance: {k + 1}/{len(sas_per_instance)} - run: {l + 1}/{num_runs}")
                results.append(execute_simann_fun(sim_ann, temp_iterations))

            resultss.append(results)
        resultsss.append(resultss)

    return problem_size, resultsss

In [69]:
from joblib import Parallel, delayed

# No logs will be printed below the cell as jupyter and joblib seem to be not very compatible regarding logging - use papermill instead if you need observability
resultssss = Parallel(n_jobs=len(problem_sizes), verbose=50)(delayed(execute_in_parallel)(i, sas_per_problem_size) for i, sas_per_problem_size in enumerate(sim_annss))


[Parallel(n_jobs=19)]: Using backend LokyBackend with 19 concurrent workers.


KeyboardInterrupt: 

In [ ]:
sorted_resultssss = sorted(resultssss, key=lambda r: r[0])
resultssss_without_index = [r[1] for r in sorted_resultssss]

In [ ]:
result_tensor = np.stack(resultssss_without_index, axis=0).squeeze()

In [ ]:
result_tensor.shape # should be a 6-tuple: (n_problem_sizes, n_load_factors_per_instance, n_instances_per_problem_size, n_runs_per_load_factor, n_temp_iter_steps, 2)

In [ ]:
np.save(results_path, result_tensor)